# Assets 3 · Gold
Owned by: Service manager. Reads from other domains only through shortcuts.

In [ ]:
# Domain: assets · owner: Service manager
# Attach this notebook to the lakehouse "Assets" in the workspace "Solvi Assets".
DOMAIN = "assets"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

In [ ]:
site = read_shared("core", "dimsite").select("site_id","municipality","country","station_id")
dim_installation = read_table("silverinstallation").join(site, "site_id").withColumn("has_battery", F.col("battery_kwh") > 0)
write_table(dim_installation, "diminstallation")
write_table(read_table("silverservicevisit"), "factservicevisit")

## Production vs. expected
Expected = kWp × irradiation × reference performance × orientation × age. Snow days are not counted as missing output.

In [ ]:
orient = spark.createDataFrame([("S",1.0),("SE",0.95),("SW",0.95),("E/W",0.84)], "orientation string, orient_factor double")
inst = dim_installation.select("installation_id","station_id","kwp","orientation","grid_connected_date").join(orient, "orientation")
wx = read_shared("core", "factweatherdaily").select("station_id","date","irradiation_kwh_m2","snow_depth_cm")
prod = (read_table("silverproductiondaily").join(inst, "installation_id").join(wx, ["station_id","date"], "left")
  .withColumn("expected_kwh", F.round(F.col("kwp") * F.col("irradiation_kwh_m2") * 0.82 * F.col("orient_factor")
                                       * (1 - 0.005 * F.datediff("date","grid_connected_date") / 365.0), 2))
  .withColumn("snow_on_panels", F.col("snow_depth_cm") > 2))
w7 = W.partitionBy("installation_id").orderBy("date").rowsBetween(-6, 0)
fact_production_daily = (prod.withColumn("kwh_7d", F.sum("kwh").over(w7))
  .withColumn("expected_7d", F.sum(F.when(~F.col("snow_on_panels"), F.col("expected_kwh"))).over(w7))
  .withColumn("ratio_7d", F.when(F.col("expected_7d") > 5, F.round(F.col("kwh_7d") / F.col("expected_7d"), 3)))
  .withColumn("underperforming", (F.col("ratio_7d") < 0.7) & ~F.col("snow_on_panels"))
  .select("installation_id","date","kwh","expected_kwh","snow_on_panels","ratio_7d","underperforming"))
write_table(fact_production_daily, "factproductiondaily")

Demo-only check against the generator's answer key.

In [ ]:
truth = read_table("silverfaultlogtruth")
flagged = read_table("factproductiondaily").filter("underperforming").select("installation_id","date")
hits = (truth.alias("t").join(flagged.alias("f"), (F.col("t.installation_id") == F.col("f.installation_id")) &
        F.col("f.date").between(F.col("t.fault_start"), F.date_add(F.col("t.fault_end"), 6)), "left")
        .groupBy("t.installation_id","t.fault_start","t.fault_type").agg((F.count("f.date") > 0).alias("detected")))
display(hits.groupBy("fault_type").agg(F.avg(F.col("detected").cast("int")).alias("detection_rate"), F.count("*").alias("faults")))